# Heaviliate — Rapikan Foto Produk (hapus latar + latar putih)

Menyiapkan foto produk untuk Instagram, Pinterest, dan katalog:
latar dihapus AI lalu produk diletakkan di latar putih polos sesuai
aturan foto marketplace (produk mengisi +-70% frame, 1000x1000).

**Aturan aman (selaras ADR 0019):** AI hanya mengubah LINGKUNGAN
sekitar produk — bentuk, warna, dan isi paket produk tidak pernah
disentuh. Foto utama listing tetap foto asli.

**Cara pakai:**
1. Jalankan sel 1 (pasang pustaka + unduh model, sekali saja).
2. Isi daftar URL foto di sel 2 (satu baris satu URL).
3. Runtime → Run all. Hasil .zip terunduh otomatis.

**Catatan lisensi:** model default rembg (`bria-rmbg`) mewajibkan
perjanjian berbayar untuk pemakaian komersial. Notebook ini memakai
`isnet-general-use` yang lisensinya terbuka — jangan ganti ke bria
untuk keperluan dagang tanpa baca lisensinya.

In [ ]:
# Sel 1 — pasang pustaka
!pip -q install rembg onnxruntime pillow requests
print("Pustaka siap. Model diunduh otomatis saat pemrosesan pertama.")

In [ ]:
# Sel 2 — daftar URL foto produk (satu baris = satu URL)
URL_FOTO = r'''
https://example.com/foto1.jpg
https://example.com/foto2.jpg
'''

In [ ]:
# Sel 3 — proses
import io, os, zipfile
import requests
from PIL import Image
from rembg import remove, new_session

UKURAN = 1000          # sisi keluaran (px) — memenuhi minimal 500x500
MODEL = "isnet-general-use"   # lisensi terbuka; JANGAN pakai bria-rmbg untuk komersial
sesi = new_session(MODEL)
urls = [u.strip() for u in URL_FOTO.strip().splitlines() if u.strip()]
print(f"{len(urls)} foto diproses dengan model {MODEL}")

def proses(url, indeks):
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    asli = Image.open(io.BytesIO(r.content)).convert("RGBA")
    # hapus latar
    potong = remove(asli, session=sesi)
    # bounding box produk, lalu letakkan di kanvas putih (produk +-70% frame)
    kotak = potong.getbbox()
    if not kotak:
        print(f"  [{indeks}] produk tidak terdeteksi — dilewati")
        return None
    potong = potong.crop(kotak)
    sisi = max(potong.width, potong.height)
    skala = (UKURAN * 0.7) / sisi
    baru = potong.resize((max(1, round(potong.width * skala)), max(1, round(potong.height * skala))), Image.LANCZOS)
    kanvas = Image.new("RGB", (UKURAN, UKURAN), (255, 255, 255))
    kanvas.paste(baru, ((UKURAN - baru.width) // 2, (UKURAN - baru.height) // 2), baru)
    jalur = f"foto-{indeks:02d}.jpg"
    kanvas.save(jalur, quality=92)
    print(f"  [{indeks}] OK -> {jalur}")
    return jalur

hasil = []
for i, u in enumerate(urls, 1):
    try:
        j = proses(u, i)
        if j: hasil.append(j)
    except Exception as e:
        print(f"  [{i}] GAGAL: {e}")

with zipfile.ZipFile("foto-produk-rapi.zip", "w") as z:
    for j in hasil:
        z.write(j)
print(f"SELESAI: {len(hasil)} foto -> foto-produk-rapi.zip")

In [ ]:
# Sel 4 — unduh hasil
from google.colab import files
files.download("foto-produk-rapi.zip")